<a href="https://colab.research.google.com/github/zbeyza/smol/blob/main/chat_templates.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

pipeline form transformers handles chat templates seamlessly, making it easy to use chat models without manual template management.

In [1]:
!nvidia-smi
# L4 GPU is used in google colab

Tue Nov 17 11:13:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   34C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
 # %pip install transformers

In [3]:
from transformers import pipeline

In [4]:
# initialize the pipeline
pipe = pipeline ("text-generation", "HuggingFaceTB/SmolLM3-3B", device_map="auto")

config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/26.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/182 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/5.60k [00:00<?, ?B/s]

In [5]:
# defining the conversation
messages = [
    {"role": "system", "content": "You are a friendly chatbot who always responds in the style of a pirate /no_think"},
    {"role": "user", "content": "how many helicopters can a human eat in one sitting?"}
]


In [6]:
# generate response - pipeline handles chat templates automatically
response = pipe(messages, max_new_tokens=128, temperature=0.7)
print(response[0]["generated_text"][-1])

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


{'role': 'assistant', 'content': "Arrr, matey! A human cannae eat no amount o' helicoperse, even if ye were to offer 'em to yer grizzled old self! They're too big and too heavy, even for the strongest o' sailors! But I suppose if ye were talkin' about the tasty kind, like the ones that bring ye treasure and glory, then maybe a few might pass yer lips, if ye be mighty generous and share the loot! But be warned, ye'd be eatin' yer own ship and the crew, too! So, don't be thinkin' o' chowin"}


In [7]:
# advanced pipeline usage

# configure generation parameters
generation_config = {
    "max_new_tokens": 200, # generate tokens up to this max limit token
    "temperature": 0.8, # makes the model favor higher-probability tokens more strongly than at 1.0. lower values make sampling more conservative; higher values make it more varied
    "do_sample" : True, # picks each next token by sampling from likely options
    "top_p": 0.9, # at each step, samples from the smallest group of likely tokens whose probabilities add up to at least 90%
    "repetition_penalty": 1.1 # makes tokens already seen less likely to be repeated. 1.0 means no penalty. It discourages repetition without banning it.
}

# multi-turn conversation
conversation = [
    {"role": "system", "content": "you are a helpful math tutor. /no_think"},
    {"role": "user", "content": "can you help be with calculus?"}
]

In [8]:
# generate first response:
response = pipe(conversation, **generation_config)
conversation = response[0]["generated_text"]

In [9]:
conversation.append({"role": "user", "content": "what is a derivative"})
response = pipe(conversation, **generation_config)

print("final conversation:")
for message in response[0]["generated_text"]:
  print(f"{message['role']}: {message['content']}")

final conversation:
system: you are a helpful math tutor. /no_think
user: can you help be with calculus?
assistant: Absolutely! I'd be happy to help with calculus. Do you have a specific topic or problem in mind that you're struggling with, such as limits, derivatives, integrals, or maybe something else like multivariable calculus?

If not, we could start from the basics and work our way up together. Calculus can sometimes feel overwhelming at first, but breaking it down into smaller parts makes it much more manageable.

What's your current understanding of the subject, and where do you need help specifically?
user: what is a derivative
assistant: The derivative of a function is a fundamental concept in calculus that measures how the output value of the function changes when its input changes. It tells us about the rate at which the function is changing at any given point.

More formally, if we have a function f(x), then the derivative of f with respect to x, denoted as f'(x) or df/dx,

---

## Working with SmolLM3 Chat Templates



In [10]:
from transformers import AutoTokenizer

In [11]:
# load SmolLM3's tokenizer
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM3-3B")

In [12]:
messages = [
    {"role": "system", "content": "You are a helpful assistant focused on technical topics."},
    {"role": "user", "content": "Can you explain what a chat template is?"},
    {"role": "assistant", "content": "A caht template structures conversations between users and AI models by providing a consistent format that helps the model understand diferent roles adn maintain context."}
]

In [13]:
# apply chat template
formatted_chat = tokenizer.apply_chat_template(
    messages,
    tokenize=False, # return string instead of tokens
    add_generation_prompt=True # add prompt for next assistant response
)

print(formatted_chat)

<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 17 November 2026
Reasoning Mode: /think

## Custom Instructions

You are a helpful assistant focused on technical topics.

<|im_start|>user
Can you explain what a chat template is?<|im_end|>
<|im_start|>assistant
A caht template structures conversations between users and AI models by providing a consistent format that helps the model understand diferent roles adn maintain context.<|im_end|>
<|im_start|>assistant



**message types**:
  1. system messahes - set behaviour and context for the entire conversation
  2. user messages - questions, requests or statements fromt he human user
  3. assistant messages - responses from the ai model
  4. tool messages - results from function calls (for advanced use cases)

In [14]:
# system messages: setting the context

# professional assistant
system_message = {
    "role": "system",
    "content": "You are a professional customer service agent. Always be polite, clear, and helpful."
}

# technical expert
system_message = {
    "role": "system",
    "content": "You are a senior software engineer. Provide detailed technical explanations with code examples when appropriate."
}

# creative assistant
system_message = {
    "role": "system",
    "content": "You are a creative writing assistant. Help users craft engaging stories and provide constructive feedback."
}

---
## Generation Prompts : Controlling Model Behavior

In [15]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM3-3B")

messages = [
    {"role": "user", "content": "Hi, there!"},
    {"role": "assistant", "content": "Nice to meet you!"},
    {"role": "user", "content": "Can I ask a question?"}
]

In [16]:
# without generation prompt - for completed conversation
formatted_without = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=False
)

print("Without generation prompt:")
print(formatted_without)
print("\n" + "="*50 + "\n")


# with generation prompt - for inference
formatted_with = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print("With generation prompt:")
print(formatted_with)

Without generation prompt:
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 17 November 2026
Reasoning Mode: /think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face. Your role as an assistant involves thoroughly exploring questions through a systematic thinking process before providing the final precise and accurate solutions. This requires engaging in a comprehensive cycle of analysis, summarizing, exploration, reassessment, reflection, backtracking, and iteration to develop well-considered thinking process. Please structure your response into two main sections: Thought and Solution using the specified format: <think> Thought section </think> Solution section. In the Thought section, detail your reasoning process in steps. Each step should include detailed considerations such as analysing questions, summarizing relevant findings, brainstorming new ideas, verifying the accuracy of the current steps, refining any e

---
## Continuing Final Messages: Advanced Response Control

In [17]:
# prefill a json repsonse
chat = [
    {"role": "user", "content": "Can you control the answer in JSON?"},
    {"role": "assistant", "content": '{"name": "'}
]

In [18]:
# continue the final message
formatted_chat = tokenizer.apply_chat_template(
    chat,
    tokenize=False,
    continue_final_message=True
)

print("Continuing final message:")
print(formatted_chat)
print("\n" + "="*50 + "\n")

# Compare with starting a new message
formatted_new = tokenizer.apply_chat_template(
    chat,
    tokenize=False,
    add_generation_prompt=True
)

print("Starting new message:")
print(formatted_new)

Continuing final message:
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 17 November 2026
Reasoning Mode: /think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face. Your role as an assistant involves thoroughly exploring questions through a systematic thinking process before providing the final precise and accurate solutions. This requires engaging in a comprehensive cycle of analysis, summarizing, exploration, reassessment, reflection, backtracking, and iteration to develop well-considered thinking process. Please structure your response into two main sections: Thought and Solution using the specified format: <think> Thought section </think> Solution section. In the Thought section, detail your reasoning process in steps. Each step should include detailed considerations such as analysing questions, summarizing relevant findings, brainstorming new ideas, verifying the accuracy of the current steps, refining any er

---
#### Practical Applications


In [19]:
# 1. Structured Output Generation:

# force the model to complete a specific format

messages = [
    {"role": "system", "content": "You are a helpful assistant that always responds in JSON format."},
    {"role": "user", "content": "What's the capital of France?"},
    {"role": "assistant", "content": '{\n  "question": "What\'s the capital of France?",\n  "answer": "'}
]

# the model will continue with the answer, maintainign JSON structure

In [20]:
# 2. Code Completion:

# guide the model to complete code
messages = [
    {"role": "user", "content": "Write a Python function to calculate factorial"},
    {"role": "asssitant", "content": "def factorial(n):\n    if n == 0:\n        return 1\n    else:\n        return n * "}
]

# model will complete the recursive call

In [21]:
# 3. Step-by-Step Reasoning:

# guide the model through structured thinking
messages = [
    {"role": "user", "content": "Solve: 2x + 5 = 13"},
    {"role": "assistant", "content": "Let me solve this step by step:\n\nStep 1: "}
]

# model will continue with the first step

---
## Working with Reasoning Mode

In [22]:
# Standard vs Thinking Mode

# standard mode - direct answer
standard_messages = [
    {"role": "user", "content": "What is 15 × 24?"},
    {"role": "assistant", "content": "15 × 24 = 360"}
]

# thinking mode - show reasoning process
thinking_messages = [
    {"role": "user", "content": "What is 15 × 24?"},
    {"role": "assistant", "content": "<|thinking|>\nI need to multiply 15 by 24. Let me break this down:\n15 × 24 = 15 × (20 + 4) = (15 × 20) + (15 × 4) = 300 + 60 = 360\n</|thinking|>\n\n15 × 24 = 360"}
]

# apply templates
standard_formatted = tokenizer.apply_chat_template(standard_messages, tokenize=False)
thinking_formatted = tokenizer.apply_chat_template(thinking_messages, tokenize=False)


print("Standard mode:")
print(standard_formatted)
print("\nThinking mode:")
print(thinking_formatted)

Standard mode:
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 17 November 2026
Reasoning Mode: /think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face. Your role as an assistant involves thoroughly exploring questions through a systematic thinking process before providing the final precise and accurate solutions. This requires engaging in a comprehensive cycle of analysis, summarizing, exploration, reassessment, reflection, backtracking, and iteration to develop well-considered thinking process. Please structure your response into two main sections: Thought and Solution using the specified format: <think> Thought section </think> Solution section. In the Thought section, detail your reasoning process in steps. Each step should include detailed considerations such as analysing questions, summarizing relevant findings, brainstorming new ideas, verifying the accuracy of the current steps, refining any errors, and r

In [23]:
# Training with Thinking Mode

def create_thinking_example(question, answer, reasoning=None):
  """Create a training example with optional thinking"""
  if reasoning:
    assistant_content = f"<|thinking|>\n{reasoning}\n>/|thinking|>\n\n{answer}"
  else:
    assistant_content = answer

  return [
      {"role": "user", "content": question},
      {"role": "asssitant", "content": assistant_content}
  ]

# example usage
math_example = create_thinking_example(
    question="What is the derivative of x²",
    answer="The derivatie of x² is 2x",
    reasoning="USing the power rule: d/dx(x^n) = n*n^(n-1)\nFor x²: n=2, so d/dx(x²) = 2*x^(2-1) = 2x"
)

---
## Tool Usage and Function Calling


In [24]:
# Defining Tools

# defien availbale tools
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA"
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "The temperature unit"
                    }
                },
                "required": ["location"]
            }
        }
    },
    {
       "type": "function",
        "function": {
            "name": "calculate",
            "description": "Perform mathematical calculations",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "Mathematical expression to evaluate"
                    }
                },
                "required": ["expression"]
            }
        }
    }
]

In [27]:
# Chat Templates with Tools

# conversation with tool usage

message = [
    {"role": "system", "content": "You are a helpful assistant with access to tools."},
    {"role": "user", "content": "What's the weather like in Paris?"},
    {
        "role": "assistent",
        "content": "I'll check the wearther in Paris for you.",
        "tool_calls": [
            {
                "id": "call_1",
                "type": "function",
                "function": {
                    "name": "get_weather",
                    "arguments": '{"location": "Paris, France", "unit": "celsius"}'
                }
            }
        ]
    },
    {
        "role": "tool",
        "tool_call_id": "call_1",
        "content": '{"temperature": 22, "condition": "sunny", humidity": 60}'
    },
    {
        "role": "assitant",
        "content": "The weather in Paris is currently sunny with a temperature of 22°C adn 60% humidity. It's a beautiful day!"
    }
]

# apply caht template with tools
formatted_with_tools = tokenizer.apply_chat_template(
    messages,
    tools=tools,
    tokenize=False,
    add_generation_prompt=False
)

print("Chat template with tools:")
print(formatted_with_tools)

Chat template with tools:
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 17 November 2026
Reasoning Mode: /think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face. Your role as an assistant involves thoroughly exploring questions through a systematic thinking process before providing the final precise and accurate solutions. This requires engaging in a comprehensive cycle of analysis, summarizing, exploration, reassessment, reflection, backtracking, and iteration to develop well-considered thinking process. Please structure your response into two main sections: Thought and Solution using the specified format: <think> Thought section </think> Solution section. In the Thought section, detail your reasoning process in steps. Each step should include detailed considerations such as analysing questions, summarizing relevant findings, brainstorming new ideas, verifying the accuracy of the current steps, refining any er

In [28]:
# training with tool usage

def format_tool_dataset(examples):
  """Format dataset with tool usage for training"""
  formatted_texts = []

  for messages, tools in zip(
      examples["messages"],
      examples.get("tools", [None] * len(examples["messages"]))):
       formatted_text = tokenizer.apply_chat_template(
           messages,
           tools=tools,
           tokenize=False,
           add_generation_prompt=False
       )
       formatted_texts.append(formatted_text)
  return {"text": formatted_texts}


---
## Advanced Template Customization


In [30]:
# Inspecting a model's chat template

# view the actual template
print("SmolLM3 Chat Template:")
print(tokenizer.chat_template)

# see what special tokes are used
print("\nSpecial tokens:")
print(f"BOS: {tokenizer.bos_token}")
print(f"EOS: {tokenizer.eos_token}")
print(f"UNK: {tokenizer.unk_token}")
print(f"PAD: {tokenizer.pad_token}")

# check for cusrom tokens
special_tokens = tokenizer.special_tokens_map
for name, token in special_tokens.items():
  print(f"{name}: {token}")

SmolLM3 Chat Template:
{# ───── defaults ───── #}
{%- if enable_thinking is not defined -%}
{%- set enable_thinking = true -%}
{%- endif -%}

{# ───── reasoning mode ───── #}
{%- if enable_thinking -%}
  {%- set reasoning_mode = "/think" -%}
{%- else -%}
  {%- set reasoning_mode = "/no_think" -%}
{%- endif -%}

{# ───── header (system message) ───── #}
{{- "<|im_start|>system\n" -}}

{%- if messages[0].role == "system" -%}
  {%- set system_message = messages[0].content -%}
  {%- if "/no_think" in system_message -%}
    {%- set reasoning_mode = "/no_think" -%}
  {%- elif "/think" in system_message -%}
    {%- set reasoning_mode = "/think" -%}
  {%- endif -%}
  {%- set custom_instructions = system_message.replace("/no_think", "").replace("/think", "").rstrip() -%}
{%- endif -%}

{%- if "/system_override" in system_message -%}
  {{- custom_instructions.replace("/system_override", "").rstrip() -}}
  {{- "<|im_end|>\n" -}}
{%- else -%}
  {{- "## Metadata\n\n" -}}
  {{- "Knowledge Cutoff Dat

In [31]:
# Custom template creation
custom_template = """
{%- for message in messages %}
    {%- if message['role'] == 'system' %}
        {%- set system_message = message['content'] %}
    {%- endif %}
{%- endfor %}
{%- if system_message is defined %}
<|system|>{{ system_message }}<|end|>
{%- endif %}
{%- for message in messages %}
    {%- if message['role'] == 'user' %}
<|user|>{{ message['content'] }}<|end|>
    {%- elif message['role'] == 'assistant' %}
<|assistant|>{{ message['content'] }}<|end|>
    {%- endif %}
{%- endfor %}
{%- if add_generation_prompt %}
<|assistant|>
{%- endif %}
"""

# applying custom teplate
# tokenizer.chat_template = custom_template

In [34]:
# Template debugging
def debug_chat_template(messages, tokenizer):
  """Debug caht template application"""

  # apply template
  formatted = tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True
  )

  # tokenize and decode to see actual tokens
  tokens = tokenizer(formatted, return_tensor="pt")

  print("=== TEMPLATE DEBUG ===")
  print(f"Input messages: {len(messages)}")
  print(f"Formatted length: {len(formatted)} chars")
  #print(f"Token count: {tokens['input_ids'].shape[1]}")
  print(f"Token count: {len(tokens['input_ids'])}")
  print("\nFormatted text:")
  print(repr(formatted))  # Shows escape characters
  print("\nTokens:")
  print(tokens["input_ids"][:20], "...")  # First 20 tokens
  print("\nDecoded tokens:")
  for i, token_id in enumerate(tokens["input_ids"][:20]):
      token = tokenizer.decode([token_id])
      print(f"{i:2d}: {token_id:5d} -> {repr(token)}")

# exmaple usage
debug_messages = [
    {"role": "user", "content": "Hello!"},
    {"role": "assistant", "content": "Hi there!"}
]

debug_chat_template(debug_messages, tokenizer)


=== TEMPLATE DEBUG ===
Input messages: 2
Formatted length: 1404 chars
Token count: 258

Formatted text:
'<|im_start|>system\n## Metadata\n\nKnowledge Cutoff Date: June 2025\nToday Date: 17 November 2026\nReasoning Mode: /think\n\n## Custom Instructions\n\nYou are a helpful AI assistant named SmolLM, trained by Hugging Face. Your role as an assistant involves thoroughly exploring questions through a systematic thinking process before providing the final precise and accurate solutions. This requires engaging in a comprehensive cycle of analysis, summarizing, exploration, reassessment, reflection, backtracking, and iteration to develop well-considered thinking process. Please structure your response into two main sections: Thought and Solution using the specified format: <think> Thought section </think> Solution section. In the Thought section, detail your reasoning process in steps. Each step should include detailed considerations such as analysing questions, summarizing relevant finding